In [1]:
%pip install python-dotenv google-genai
%pip install -U "langchain[google-genai]"
%pip install -U langchain-community pypdf
%pip install chromadb
%pip install -U sentence-transformers langchain-huggingface


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Test cell to check connection to LLM
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from .env file
load_dotenv()

# Get the API key from the environment variable
# The .env file has the API KEY.
api_key = os.getenv("GEMINI_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

response = model.invoke("Do parrots talk")
print(response)

/home/rvv/Downloads/Resumizer/gemini-env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/home/rvv/Downloads/Resumizer/gemini-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='Yes, **parrots can talk**, or more accurately, they can **mimic human speech and other sounds**.\n\nHere\'s a breakdown of what that means:\n\n*   **Mimicry, not understanding:** Parrots don\'t understand the meaning of the words they say in the same way humans do. They are incredibly adept at learning and reproducing sounds they hear frequently.\n*   **Vocal learning:** Parrots are among the few bird species that are vocal learners. This means they learn their vocalizations by listening to others and practicing.\n*   **Variety of sounds:** Beyond human speech, parrots can mimic a wide range of sounds, including:\n    *   Other animal sounds (dogs barking, cats meowing)\n    *   Household noises (doorbells, telephones, alarms)\n    *   Music\n    *   Even other people\'s voices.\n*   **Intelligence and social interaction:** While they don\'t understand meaning, their ability to mimic is often linked to their intelligence and their strong social nature. They often learn to "tal

In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
# Path to your main folder
path = "./data/data/data/ENGINEERING"

# glob="**/*.pdf" tells it to look in all subfolders recursively
loader = DirectoryLoader(
    path, 
    glob="**/*.pdf", 
    loader_cls=PyPDFLoader,
    show_progress=True, # Shows a handy progress bar
    use_multithreading=True # Speeds up loading for large volumes
)

docs = loader.load()
len(docs)

100%|██████████████████████████████████████████████████████████████| 118/118 [00:22<00:00,  5.31it/s]


227

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

chunks = text_splitter.split_documents(docs)

In [14]:
from google import genai

client = genai.Client(api_key=api_key)

result = client.models.embed_content(
        model="gemini-embedding-001",
        contents="What is the meaning of life?"
)

print(result.embeddings)

[ContentEmbedding(
  values=[
    -0.022374554,
    -0.004560777,
    0.013309286,
    -0.0545072,
    -0.02090443,
    <... 3067 more items ...>,
  ]
)]


In [42]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [43]:
# 950 chunks 
# Google Free Teir for embedding model has following rate limits
# RPM - Requests per minute - 100
# RPD - Request per days - 1000
# TPM - Tokens per minute - 30K 

len(chunks)

import time
from langchain_community.vectorstores import Chroma

# This does not work since we hit HTTP 429 rate limit.
# vectorstore = Chroma.from_documents(
#     documents=chunks, 
#     embedding=embeddings,
#     persist_directory="./chroma_db"
# )

# Work around to still get around limit.
# Initialize an empty vectorstore first
vectorstore = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=embeddings
)

# Add documents in small chunks with a pause
batch_size = 10 
for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Uploaded {i + len(batch)} / {len(chunks)} chunks...")
    
    # This 5-second sleep is the "magic" that prevents the 429 error
    time.sleep(10)

Uploaded 10 / 950 chunks...
Uploaded 20 / 950 chunks...
Uploaded 30 / 950 chunks...
Uploaded 40 / 950 chunks...
Uploaded 50 / 950 chunks...
Uploaded 60 / 950 chunks...
Uploaded 70 / 950 chunks...
Uploaded 80 / 950 chunks...
Uploaded 90 / 950 chunks...
Uploaded 100 / 950 chunks...
Uploaded 110 / 950 chunks...
Uploaded 120 / 950 chunks...
Uploaded 130 / 950 chunks...
Uploaded 140 / 950 chunks...
Uploaded 150 / 950 chunks...
Uploaded 160 / 950 chunks...
Uploaded 170 / 950 chunks...
Uploaded 180 / 950 chunks...
Uploaded 190 / 950 chunks...
Uploaded 200 / 950 chunks...
Uploaded 210 / 950 chunks...
Uploaded 220 / 950 chunks...
Uploaded 230 / 950 chunks...
Uploaded 240 / 950 chunks...
Uploaded 250 / 950 chunks...
Uploaded 260 / 950 chunks...
Uploaded 270 / 950 chunks...
Uploaded 280 / 950 chunks...
Uploaded 290 / 950 chunks...
Uploaded 300 / 950 chunks...
Uploaded 310 / 950 chunks...
Uploaded 320 / 950 chunks...
Uploaded 330 / 950 chunks...
Uploaded 340 / 950 chunks...
Uploaded 350 / 950 chun

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 7.801056014s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'qu

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os
import shutil

# 1. FORCE DELETE the old folder to be 100% sure
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
# "all-MiniLM-L6-v2" is fast, small, and very accurate for its size
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
abs_path = os.path.abspath("./chroma_db")
# Now use it exactly like you did before
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory=abs_path
)

Loading weights: 100%|███████████████████████████████████████████| 103/103 [00:00<00:00, 5524.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# 3. Verify it worked
count = vectorstore._collection.count()
if(len(chunks)==count)):
    print("The 

950
950
